In [3]:
from pathlib import Path

project_dir = Path.home() / "Downloads" / "SCADA Water Analytics"
raw_dir = project_dir

print("Project folder exists:", project_dir.exists())
print("CSV files found:", len(list(raw_dir.glob("*.csv"))))

for file in sorted(raw_dir.glob("*.csv")):
    print(file.name)

Project folder exists: True
CSV files found: 10
2018_Leakages.csv
2018_SCADA_Demands.csv
2018_SCADA_Flows.csv
2018_SCADA_Levels.csv
2018_SCADA_Pressures.csv
2019_Leakages.csv
2019_SCADA_Demands.csv
2019_SCADA_Flows.csv
2019_SCADA_Levels.csv
2019_SCADA_Pressures.csv


In [21]:
import pandas as pd
demands_2018 = pd.read_csv(r'C:\Users\HP\Downloads\SCADA Water Analytics\2018_SCADA_Demands.csv')

ParserError: Error tokenizing data. C error: Expected 81 fields in line 3, saw 83


In [22]:
#Inspect the first five lines of the 2018_SCADA_Demands.csv
# This displays the first five lines, including commas and quotation marks, so we can spot formatting issues.

from pathlib import Path

file_path = project_dir / "2018_SCADA_Demands.csv"

with open(file_path, "r", encoding="utf-8-sig", errors="replace") as f:
    for line_number in range(1, 6):
        line = f.readline()
        if not line:
            break
        print(f"\n--- Line {line_number} ---")
        print(repr(line[:1000]))


--- Line 1 ---
'Timestamp;n1;n2;n3;n4;n6;n7;n8;n9;n10;n11;n13;n16;n17;n18;n19;n20;n21;n22;n23;n24;n25;n26;n27;n28;n29;n30;n31;n32;n33;n34;n35;n36;n39;n40;n41;n42;n43;n44;n45;n343;n344;n345;n346;n347;n349;n350;n351;n352;n353;n354;n355;n356;n357;n358;n360;n361;n362;n364;n365;n366;n367;n368;n369;n370;n371;n372;n373;n374;n375;n376;n377;n378;n379;n381;n382;n383;n384;n385;n386;n387;n388;n389\n'

--- Line 2 ---
'2018-01-01 00:00:00;40,83;112,23;96,71;131,29;166,18;81,92;249,95;489,63;58,83;495,98;266,97;99,39;251,8;70,03;85,34;48,12;53,85;135,47;194,87;39,39;989,73;32,63;66,15;540,76;65,42;134,28;99,56;67,56;103,78;0;41,65;210,53;10,8;323,74;74,67;111,3;19,73;24,96;98,54;94,25;120,71;41,93;200,02;0,64;27,24;83,99;329,38;93,91;359,98;126,69;0;118,26;90,4;241,93;250,67;137,42;6,22;18,58;114,43;56,9;77,96;75,7;96,52;206,12;85,18;47,01;10,37;16,38;60,49;58,27;318,46;4,35;33,41;150,82;67,61;65,82;65,25;57,39;214,2;81,49;256,5;92,14\n'

--- Line 3 ---
'2018-01-01 00:05:00;40,2;166,05;239,53;136,21

In [23]:
# Check the field counts

import csv

with open(file_path, "r", encoding="utf-8-sig", newline="") as f:
    reader = csv.reader(f)
    for row_number, row in zip(range(1, 6), reader):
        print(f"Line {row_number}: {len(row)} fields")

Line 1: 1 fields
Line 2: 81 fields
Line 3: 83 fields
Line 4: 82 fields
Line 5: 78 fields


In [ ]:
# We've found the cause. Your 2018_SCADA_Demands.csv uses semicolons to separate columns and commas as decimal marks. 
# For example: Timestamp;n1;n2;n3;... — semicolons separate the columns.
# 40,83 — the comma represents the decimal in the value 40.83.
# Your original code used pd.read_csv(path), which assumes comma-separated columns by default. 
# That makes Pandas split decimal values such as 40,83 into separate fields, causing the parser error.
# The file's inconsistent field counts are therefore explained by the wrong delimiter. 
# We should read it using sep=";" and decimal="," instead.

In [24]:
# LOAD THE DEMANDS_2018 FILE CORRECTLY
import pandas as pd

file_path = r'C:\Users\HP\Downloads\SCADA Water Analytics\2018_SCADA_Demands.csv'
demands_2018 = pd.read_csv(
    file_path,
    sep=';',
    decimal=','
)

In [25]:
demands_2018.head(3)

,Timestamp,n1,n2,n3,n4,n6,n7,n8,n9,n10,...,n379,n381,n382,n383,n384,n385,n386,n387,n388,n389
0,2018-01-01 00:00:00,40.83,112.23,96.71,131.29,166.18,81.92,249.95,489.63,58.83,...,33.41,150.82,67.61,65.82,65.25,57.39,214.20,81.49,256.50,92.14
1,2018-01-01 00:05:00,40.20,166.05,239.53,136.21,154.48,372.14,190.56,171.60,38.02,...,78.90,97.46,95.29,50.18,81.29,49.67,109.14,104.46,394.48,203.61
2,2018-01-01 00:10:00,38.06,121.91,69.12,131.00,160.10,320.79,242.07,39.35,52.79,...,100.26,135.33,178.10,63.74,252.94,9.69,285.82,94.32,337.33,93.11


In [26]:
demands_2018.shape

# 105120 rows and 83 columns

(105120, 83)

In [27]:
demands_2018.dtypes

Timestamp        str
n1           float64
n2           float64
n3           float64
n4           float64
              ...   
n385         float64
n386         float64
n387         float64
n388         float64
n389         float64
Length: 83, dtype: object

In [57]:
# Convert the Timestamp dtype to datetime
demands_2018["Timestamp"] = pd.to_datetime(
    demands_2018["Timestamp"],
    errors="coerce"
)

In [58]:
demands_2018.dtypes

Timestamp    datetime64[us]
n1                  float64
n2                  float64
n3                  float64
n4                  float64
                  ...      
n385                float64
n386                float64
n387                float64
n388                float64
n389                float64
Length: 83, dtype: object

In [50]:
demands_2018.isnull().sum()

Timestamp    0
n1           0
n2           0
n3           0
n4           0
            ..
n385         0
n386         0
n387         0
n388         0
n389         0
Length: 83, dtype: int64

In [ ]:
# 2018 SCADA Demands file has no missing values

In [56]:
demands_2018.duplicated().sum()

np.int64(0)

In [ ]:
# 2018 SCADA Demands file has no duplicated values

In [28]:
# LOAD LEVELS_2018

import pandas as pd

file_path = r'C:\Users\HP\Downloads\SCADA Water Analytics\2018_SCADA_Levels.csv'

levels_2018 = pd.read_csv(
    file_path,
    sep=';',
    decimal=','
)

In [29]:
levels_2018.head()

,Timestamp,T1
0,2018-01-01 00:00:00,3.50
1,2018-01-01 00:05:00,3.51
2,2018-01-01 00:10:00,3.53
3,2018-01-01 00:15:00,3.54
4,2018-01-01 00:20:00,3.56


In [30]:
levels_2018.shape

# 105120 rows and 2 columns

(105120, 2)

In [31]:
levels_2018.dtypes

Timestamp        str
T1           float64
dtype: object

In [59]:
# Convert the Timestamp dtype to datetime
levels_2018["Timestamp"] = pd.to_datetime(
    levels_2018["Timestamp"],
    errors="coerce"
)

In [60]:
levels_2018.dtypes

Timestamp    datetime64[us]
T1                  float64
dtype: object

In [49]:
levels_2018.isnull().sum()

Timestamp    0
T1           0
dtype: int64

In [ ]:
# 2018 SCADA Levels file has no missing values

In [55]:
levels_2018.duplicated().sum()

np.int64(0)

In [ ]:
# 2018 SCADA Levels file has no duplicated values

In [32]:
# LOAD PRESSURES_2018

import pandas as pd

file_path = r'C:\Users\HP\Downloads\SCADA Water Analytics\2018_SCADA_Pressures.csv'

pressures_2018 = pd.read_csv(
    file_path,
    sep=';',
    decimal=','
)

In [33]:
pressures_2018.head()

,Timestamp,n1,n4,n31,n54,n105,n114,n163,n188,n215,...,n549,n613,n636,n644,n679,n722,n726,n740,n752,n769
0,2018-01-01 00:00:00,28.92,33.87,37.13,37.13,50.52,53.99,52.56,55.32,39.09,...,54.80,56.16,45.53,47.62,47.33,46.14,47.12,43.81,49.17,48.50
1,2018-01-01 00:05:00,28.94,33.89,37.16,37.23,50.60,54.09,52.67,55.42,39.10,...,54.90,56.26,45.61,47.71,47.39,46.19,47.17,43.83,49.22,48.54
2,2018-01-01 00:10:00,28.96,33.91,37.18,37.21,50.57,54.05,52.64,55.39,39.09,...,54.87,56.24,45.61,47.71,47.41,46.20,47.19,43.84,49.24,48.56
3,2018-01-01 00:15:00,28.98,33.93,37.19,37.27,50.60,54.08,52.70,55.45,39.09,...,54.92,56.30,45.65,47.75,47.43,46.21,47.21,43.84,49.26,48.57
4,2018-01-01 00:20:00,28.98,33.93,37.19,37.14,50.53,54.00,52.57,55.32,39.09,...,54.80,56.17,45.54,47.64,47.34,46.15,47.13,43.81,49.17,48.50


In [35]:
pressures_2018.shape

# 105120 rows and 34 columns

(105120, 34)

In [36]:
pressures_2018.dtypes

Timestamp        str
n1           float64
n4           float64
n31          float64
n54          float64
n105         float64
n114         float64
n163         float64
n188         float64
n215         float64
n229         float64
n288         float64
n296         float64
n332         float64
n342         float64
n410         float64
n415         float64
n429         float64
n458         float64
n469         float64
n495         float64
n506         float64
n516         float64
n519         float64
n549         float64
n613         float64
n636         float64
n644         float64
n679         float64
n722         float64
n726         float64
n740         float64
n752         float64
n769         float64
dtype: object

In [61]:
# Convert the Timestamp dtype to datetime
pressures_2018["Timestamp"] = pd.to_datetime(
    pressures_2018["Timestamp"],
    errors="coerce"
)

In [62]:
pressures_2018.dtypes

Timestamp    datetime64[us]
n1                  float64
n4                  float64
n31                 float64
n54                 float64
n105                float64
n114                float64
n163                float64
n188                float64
n215                float64
n229                float64
n288                float64
n296                float64
n332                float64
n342                float64
n410                float64
n415                float64
n429                float64
n458                float64
n469                float64
n495                float64
n506                float64
n516                float64
n519                float64
n549                float64
n613                float64
n636                float64
n644                float64
n679                float64
n722                float64
n726                float64
n740                float64
n752                float64
n769                float64
dtype: object

In [48]:
pressures_2018.isnull().sum()

Timestamp    0
n1           0
n4           0
n31          0
n54          0
n105         0
n114         0
n163         0
n188         0
n215         0
n229         0
n288         0
n296         0
n332         0
n342         0
n410         0
n415         0
n429         0
n458         0
n469         0
n495         0
n506         0
n516         0
n519         0
n549         0
n613         0
n636         0
n644         0
n679         0
n722         0
n726         0
n740         0
n752         0
n769         0
dtype: int64

In [ ]:
# 2018 SCADA Pressures file has no missing values

In [54]:
pressures_2018.duplicated().sum()

np.int64(0)

In [ ]:
# 2018 SCADA Pressures file has no duplicated values

In [37]:
# LOAD FLOWS_2018

import pandas as pd

file_path = r'C:\Users\HP\Downloads\SCADA Water Analytics\2018_SCADA_Flows.csv'

flows_2018 = pd.read_csv(
    file_path,
    sep=';',
    decimal=','
)

In [38]:
flows_2018.head()

,Timestamp,p227,p235,PUMP_1
0,2018-01-01 00:00:00,77.77,83.93,44.04
1,2018-01-01 00:05:00,72.51,76.34,44.06
2,2018-01-01 00:10:00,71.54,78.68,44.06
3,2018-01-01 00:15:00,69.85,75.50,44.07
4,2018-01-01 00:20:00,77.12,82.99,44.03


In [39]:
flows_2018.shape

# 105120 rows and 4 columns

(105120, 4)

In [40]:
flows_2018.dtypes

Timestamp        str
p227         float64
p235         float64
PUMP_1       float64
dtype: object

In [63]:
# Convert the Timestamp dtype to datetime
flows_2018["Timestamp"] = pd.to_datetime(
    flows_2018["Timestamp"],
    errors="coerce"
)

In [64]:
flows_2018.dtypes

Timestamp    datetime64[us]
p227                float64
p235                float64
PUMP_1              float64
dtype: object

In [47]:
flows_2018.isnull().sum()

Timestamp    0
p227         0
p235         0
PUMP_1       0
dtype: int64

In [ ]:
# 2018_SCADA_FLOWS FILE HAS NO MISSING VALUES

In [53]:
flows_2018.duplicated().sum()

np.int64(0)

In [ ]:
# 2018_SCADA_FLOWS FILE HAS NO DUPLICATED VALUES

In [41]:
# load the 2018 LEAKAGES FILE

import pandas as pd

file_path = r'C:\Users\HP\Downloads\SCADA Water Analytics\2018_Leakages.csv'

leakages_2018 = pd.read_csv(
    file_path,
    sep=';',
    decimal=','
)

In [42]:
leakages_2018.head()

,Timestamp,p31,p158,p183,p232,p257,p369,p427,p461,p538,p628,p654,p673,p810,p866
0,2018-01-01 00:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2018-01-01 00:05:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2018-01-01 00:10:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2018-01-01 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2018-01-01 00:20:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [43]:
leakages_2018.shape

# 105120 rows and 15 columns

(105120, 15)

In [44]:
leakages_2018.dtypes

Timestamp        str
p31          float64
p158         float64
p183         float64
p232         float64
p257         float64
p369         float64
p427         float64
p461         float64
p538         float64
p628         float64
p654         float64
p673         float64
p810         float64
p866         float64
dtype: object

In [65]:
# Convert the Timestamp dtype to datetime
leakages_2018["Timestamp"] = pd.to_datetime(
    leakages_2018["Timestamp"],
    errors="coerce"
)

In [66]:
leakages_2018.dtypes

Timestamp    datetime64[us]
p31                 float64
p158                float64
p183                float64
p232                float64
p257                float64
p369                float64
p427                float64
p461                float64
p538                float64
p628                float64
p654                float64
p673                float64
p810                float64
p866                float64
dtype: object

In [46]:
leakages_2018.isnull().sum()

Timestamp    0
p31          0
p158         0
p183         0
p232         0
p257         0
p369         0
p427         0
p461         0
p538         0
p628         0
p654         0
p673         0
p810         0
p866         0
dtype: int64

In [ ]:
# 2018_Leakages file has no missing values

In [52]:
leakages_2018.duplicated().sum()

np.int64(0)

In [ ]:
# 2018_Leakages file has no duplicated values

In [67]:
# Export the cleaned leakages_2018 file as a standard CSV file
output_path = 'cleaned_leakages_2018.csv'
leakages_2018.to_csv(output_path, index=False, encoding='utf-8')
print()

In [68]:
# Export the cleaned Pressures_2018 file as a standard CSV file
output_path = 'cleaned_pressures_2018.csv'
pressures_2018.to_csv(output_path, index=False, encoding='utf-8')
print()

In [69]:
# Export the cleaned levels_2018 file as a standard CSV file
output_path = 'cleaned_levels_2018.csv'
levels_2018.to_csv(output_path, index=False, encoding='utf-8')
print()

In [70]:
# Export the cleaned demands_2018 file as a standard CSV file
output_path = 'cleaned_demands_2018.csv'
demands_2018.to_csv(output_path, index=False, encoding='utf-8')
print()

In [71]:
# Export the cleaned flows_2018 file as a standard CSV file
output_path = 'cleaned_flows_2018.csv'
flows_2018.to_csv(output_path, index=False, encoding='utf-8')
print()